In [0]:
import requests
import time

companies=['GOOGL','MSFT','AAPL','TSLA']
APIKEY='F3QZ1JCRX9IY64KF'
records=[]

for company in companies:
    url=f"https://www.alphavantage.co/query?function=TIME_SERIES_DAILY&symbol={company}&apikey={APIKEY}"
    try:
        r=requests.get(url)
        data=r.json()

        for date, values in data['Time Series (Daily)'].items():
            records.append([company, date, values['1. open'], values['2. high'], values['3. low'], values['4. close'], values['5. volume']])
        
    except Exception as err:
        print(f"Error for {company}: {err}")
    
    time.sleep(30)

In [0]:
df=spark.createDataFrame(records,schema=['symbol','date','open','high','low','close','volume'])
display(df)

symbol,date,open,high,low,close,volume
GOOGL,2026-08-06,360.7700,364.1259,356.8800,357.7500,24982494
GOOGL,2026-08-05,383.3400,384.4800,356.7700,362.4300,46926339
GOOGL,2026-08-04,368.2200,380.5200,367.5000,377.6500,35705703
GOOGL,2026-08-03,365.4450,376.6932,363.3500,373.5100,38669652
GOOGL,2026-07-31,340.8300,358.5800,340.0000,356.1300,46498023
GOOGL,2026-07-30,334.1950,336.5100,330.3400,333.6600,30168000
GOOGL,2026-07-29,334.6650,342.5000,331.6200,336.7100,27501632
GOOGL,2026-07-28,327.9000,335.8800,324.4400,333.7100,29511940
GOOGL,2026-07-27,325.0400,330.4200,324.4500,326.5600,28471593
GOOGL,2026-07-24,318.4200,324.1800,317.3200,319.7400,31806553


In [0]:
df.write.mode('overwrite').saveAsTable('databricks_fundamentals.bronze.daily_stock_prices')

# Silver Processing

In [0]:
import pyspark.sql.functions as F
bronze=spark.table('databricks_fundamentals.bronze.daily_stock_prices')
display(bronze)

symbol,date,open,high,low,close,volume
GOOGL,2026-08-06,360.7700,364.1259,356.8800,357.7500,24982494
GOOGL,2026-08-05,383.3400,384.4800,356.7700,362.4300,46926339
GOOGL,2026-08-04,368.2200,380.5200,367.5000,377.6500,35705703
GOOGL,2026-08-03,365.4450,376.6932,363.3500,373.5100,38669652
GOOGL,2026-07-31,340.8300,358.5800,340.0000,356.1300,46498023
GOOGL,2026-07-30,334.1950,336.5100,330.3400,333.6600,30168000
GOOGL,2026-07-29,334.6650,342.5000,331.6200,336.7100,27501632
GOOGL,2026-07-28,327.9000,335.8800,324.4400,333.7100,29511940
GOOGL,2026-07-27,325.0400,330.4200,324.4500,326.5600,28471593
GOOGL,2026-07-24,318.4200,324.1800,317.3200,319.7400,31806553


In [0]:
display(bronze.describe())

summary,symbol,date,open,high,low,close,volume
count,400,400,400,400,400,400,400
mean,null,null,357.0552775,362.37434725000014,352.11714374999997,357.4368999999999,4.38139172175E7
stddev,null,null,52.56241443158493,53.88632336392135,51.39397730920335,52.74895447670945,2.199818015137505E7
min,AAPL,2026-03-16,247.9100,249.1999,245.5100,246.6300,107253659
max,TSLA,2026-08-06,496.3550,501.5550,488.5200,499.8600,93969510


In [0]:

silver=bronze.withColumn('date',F.to_timestamp(F.col('date'))).withColumn('open',F.round(F.col('open').cast('double'),2)).withColumn('high',F.round(F.col('high').cast('double'),2)).withColumn('low',F.round(F.col('low').cast('double'),2)).withColumn('close',F.round(F.col('close').cast('double'),2)).withColumn('volume',F.col('volume').cast('int'))
display(silver)

symbol,date,open,high,low,close,volume
GOOGL,2026-08-06T00:00:00.000Z,360.77,364.13,356.88,357.75,24982494
GOOGL,2026-08-05T00:00:00.000Z,383.34,384.48,356.77,362.43,46926339
GOOGL,2026-08-04T00:00:00.000Z,368.22,380.52,367.5,377.65,35705703
GOOGL,2026-08-03T00:00:00.000Z,365.45,376.69,363.35,373.51,38669652
GOOGL,2026-07-31T00:00:00.000Z,340.83,358.58,340.0,356.13,46498023
GOOGL,2026-07-30T00:00:00.000Z,334.2,336.51,330.34,333.66,30168000
GOOGL,2026-07-29T00:00:00.000Z,334.67,342.5,331.62,336.71,27501632
GOOGL,2026-07-28T00:00:00.000Z,327.9,335.88,324.44,333.71,29511940
GOOGL,2026-07-27T00:00:00.000Z,325.04,330.42,324.45,326.56,28471593
GOOGL,2026-07-24T00:00:00.000Z,318.42,324.18,317.32,319.74,31806553


# Gold Processing

In [0]:
latestDate=silver.agg(F.max(F.col('date')))
display(latestDate.first()[0])


datetime.datetime(2026, 8, 4, 0, 0)

In [0]:
from pyspark.sql import Window
window=Window.partitionBy('symbol').orderBy('date')

gold = silver.withColumn('price_change_5d', F.col('close') - F.lag(F.col('close'), 5).over(window)).withColumn('price_change_30d', F.col('close') - F.lag(F.col('close'), 30).over(window)).withColumn('price_change_90d', F.col('close') - F.lag(F.col('close'), 90).over(window)).withColumn('percentage_change_5d', F.round((F.col('price_change_5d') / F.lag(F.col('close'), 5).over(window)*100), 2)).withColumn('percentage_change_30d', F.round((F.col('price_change_30d') / F.lag(F.col('close'), 30).over(window))*100, 2)).withColumn('percentage_change_90d', F.round((F.col('price_change_90d') / F.lag(F.col('close'), 90).over(window))*100, 2)).withColumn('volume_change_5d', F.col('volume') - F.lag(F.col('volume'), 5).over(window)).withColumn('volume_change_30d', F.col('volume') - F.lag(F.col('volume'), 30).over(window)).withColumn('volume_change_90d', F.col('volume') - F.lag(F.col('volume'), 90).over(window))

display(gold)

symbol,date,open,high,low,close,volume,price_change_5d,price_change_30d,price_change_90d,percentage_change_5d,percentage_change_30d,percentage_change_90d,volume_change_5d,volume_change_30d,volume_change_90d
AAPL,2026-03-16T00:00:00.000Z,252.11,253.89,249.88,252.82,32074209,null,null,null,null,null,null,null,null,null
AAPL,2026-03-17T00:00:00.000Z,252.96,255.13,252.18,254.23,32361607,null,null,null,null,null,null,null,null,null
AAPL,2026-03-18T00:00:00.000Z,252.63,254.94,249.0,249.94,35757874,null,null,null,null,null,null,null,null,null
AAPL,2026-03-19T00:00:00.000Z,249.4,251.83,247.3,248.96,34864082,null,null,null,null,null,null,null,null,null
AAPL,2026-03-20T00:00:00.000Z,247.98,249.2,246.0,247.99,88331081,null,null,null,null,null,null,null,null,null
AAPL,2026-03-23T00:00:00.000Z,253.97,254.6,250.28,251.49,40546109,-1.329999999999984,null,null,-0.53,null,null,8471900,null,null
AAPL,2026-03-24T00:00:00.000Z,250.35,254.83,249.55,251.64,45152288,-2.5900000000000034,null,null,-1.02,null,null,12790681,null,null
AAPL,2026-03-25T00:00:00.000Z,254.1,255.0,251.6,252.62,28476668,2.680000000000007,null,null,1.07,null,null,-7281206,null,null
AAPL,2026-03-26T00:00:00.000Z,252.12,257.0,250.77,252.89,41796650,3.9299999999999784,null,null,1.58,null,null,6932568,null,null
AAPL,2026-03-27T00:00:00.000Z,253.9,255.49,248.07,248.8,47899998,0.8100000000000023,null,null,0.33,null,null,-40431083,null,null
